# 9. Scaled Benchmark: Vector RAG vs Graph RAG on GraphRAG-Bench (Medical)

**Presentation map:** Slide 33 (graph construction: chunk → LLM extracts entities + relations), Slide 34 (the *indexing price*),
Slide 36 (graphs help on reasoning and summarisation, not on fact lookup — source: *When to use Graphs in RAG*,
arXiv 2506.05690), Slide 37 (indexing cost differs by an order of magnitude), Slide 38 (decision rule by question type).

Notebooks 1 and 6 make the slide-36 point on small, hand-built question sets. This notebook re-tests it on the
**public benchmark behind the slide**: GraphRAG-Bench (`GraphRAG-Bench/GraphRAG-Bench` on Hugging Face, MIT licence) —
4,072 questions over a medical and a novel corpus, labelled *Fact Retrieval*, *Complex Reasoning*,
*Contextual Summarize* and *Creative Generation*.

**Design**
- **Corpus:** the complete Medical corpus (≈1.05 M characters, NCCN-style patient guidelines), split into ~600-token chunks.
- **Questions:** a fixed-seed random sample of 50 per type (Fact, Reasoning, Summarize) + 30 Creative = 180 of the 2,062 medical questions.
- **(a) Vector RAG:** `text-embedding-3-small`, top-8 chunks → `gpt-4o-mini`.
- **(b) Graph RAG (HippoRAG-2-style):** `gpt-4o-mini` extracts entities and relations from every chunk (structured
  output); entities are merged by normalised name into one entity–chunk graph with provenance; at query time the
  question's entities are linked by embedding to graph entities, **personalised PageRank** spreads from them (plus a
  small weight on dense chunk scores, as in HippoRAG 2) to rank chunks; the answer sees the top-5 chunks + the
  relations among the top-ranked entities.
- **Scoring — the benchmark's own metrics**, re-implemented from `Evaluation/` of the dataset repo:
  *answer correctness* = 0.75 × statement-level F1 (LLM classifies TP/FP/FN against the gold answer) + 0.25 × embedding
  similarity, for every type; *ROUGE-L* for Fact/Reasoning; *coverage* (share of gold facts present) for
  Summarize/Creative. Same judge (`gpt-4o-mini`, temperature 0) for both systems; `text-embedding-3-small` replaces the
  repo's BGE embedder.

Everything (chunks, extractions, embeddings, answers, scores) is cached under `data/benchmarks/graphrag_bench/cache/`, so the
notebook re-runs in seconds; set `RUN_LIVE = True` to recompute from scratch (≈15 min, a few million tokens).

In [1]:
import json, os, re, time, random, hashlib
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import networkx as nx
from pydantic import BaseModel, Field
from openai import OpenAI

RUN_LIVE = False                      # True: ignore the cache and recompute everything
BASE = "data/benchmarks/graphrag_bench"
CACHE = f"{BASE}/cache"
os.makedirs(CACHE, exist_ok=True)

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
CHAT, EMBED = "gpt-4o-mini", "text-embedding-3-small"
USAGE = defaultdict(lambda: {"calls": 0, "in": 0, "out": 0})   # token ledger per phase

def _retry(fn, tries=6):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:                       # rate limits / transient errors: back off
            if i == tries - 1:
                raise
            time.sleep(2 ** i + random.random())

def chat(messages, phase, fmt=None, max_tokens=None):
    kw = dict(model=CHAT, messages=messages, temperature=0, seed=7)
    if max_tokens:
        kw["max_tokens"] = max_tokens
    if fmt is None:
        r = _retry(lambda: client.chat.completions.create(**kw))
        out = r.choices[0].message.content
    else:
        r = _retry(lambda: client.beta.chat.completions.parse(response_format=fmt, **kw))
        out = r.choices[0].message.parsed
    u = USAGE[phase]; u["calls"] += 1
    u["in"] += r.usage.prompt_tokens; u["out"] += r.usage.completion_tokens
    return out

def embed(texts, phase, batch=256):
    out = []
    for i in range(0, len(texts), batch):
        r = _retry(lambda: client.embeddings.create(model=EMBED, input=texts[i:i + batch]))
        out += [d.embedding for d in r.data]
        USAGE[phase]["calls"] += 1; USAGE[phase]["in"] += r.usage.prompt_tokens
    v = np.array(out, dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

def cached_json(name, compute):
    path = f"{CACHE}/{name}"
    if os.path.exists(path) and not RUN_LIVE:
        return json.load(open(path))
    obj = compute(); json.dump(obj, open(path, "w")); return obj

def cached_npy(name, compute):
    path = f"{CACHE}/{name}"
    if os.path.exists(path) and not RUN_LIVE:
        return np.load(path)
    v = compute(); np.save(path, v); return v

def pmap(fn, items, workers=16):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(fn, items))

# ---- load the benchmark -----------------------------------------------------
corpus = json.load(open(f"{BASE}/Datasets/Corpus/medical.json"))[0]["context"]
questions = json.load(open(f"{BASE}/Datasets/Questions/medical_questions.json"))
novel_n = len(json.load(open(f"{BASE}/Datasets/Questions/novel_questions.json")))
print(f"GraphRAG-Bench: {len(questions):,} medical + {novel_n:,} novel = {len(questions) + novel_n:,} questions")
print("Medical question types:", dict(Counter(q["question_type"] for q in questions)))
print(f"Medical corpus: {len(corpus):,} characters (≈{len(corpus) // 4:,} tokens)")

TYPES = {"Fact Retrieval": 50, "Complex Reasoning": 50, "Contextual Summarize": 50, "Creative Generation": 30}
rng = random.Random(7)
sample = []
for t, n in TYPES.items():
    pool = sorted((q for q in questions if q["question_type"] == t), key=lambda q: q["id"])
    sample += rng.sample(pool, n)
print(f"Sampled {len(sample)} questions:", dict(Counter(q["question_type"] for q in sample)))

GraphRAG-Bench: 2,062 medical + 2,010 novel = 4,072 questions
Medical question types: {'Fact Retrieval': 1098, 'Complex Reasoning': 509, 'Contextual Summarize': 289, 'Creative Generation': 166}
Medical corpus: 1,052,159 characters (≈263,039 tokens)
Sampled 180 questions: {'Fact Retrieval': 50, 'Complex Reasoning': 50, 'Contextual Summarize': 50, 'Creative Generation': 30}


## Index (a): chunks + dense embeddings — the vector baseline's whole indexing cost

In [2]:
CHUNK_CHARS, OVERLAP = 2400, 300        # ≈600 tokens per chunk, ≈75-token overlap

def make_chunks(text):
    out, i = [], 0
    while i < len(text):
        j = min(len(text), i + CHUNK_CHARS)
        if j < len(text):                 # end on a sentence boundary when possible
            k = text.rfind(". ", i + CHUNK_CHARS // 2, j)
            j = k + 1 if k > 0 else j
        out.append(text[i:j].strip())
        if j >= len(text):
            break
        i = j - OVERLAP
    return out

chunks = cached_json("chunks.json", lambda: make_chunks(corpus))
t0 = time.time()
chunk_vecs = cached_npy("chunk_vecs.npy", lambda: embed(chunks, "vector index"))
print(f"{len(chunks)} chunks, embeddings {chunk_vecs.shape}  ({time.time() - t0:.1f} s, cached={not RUN_LIVE})")

519 chunks, embeddings (519, 1536)  (0.0 s, cached=True)


## Index (b): LLM entity/relation extraction → entity–chunk graph (Slide 33)

One structured-output call per chunk. Entities are merged by normalised name (cheapest stage of the slide-48 cascade);
every entity and relation keeps the ids of the chunks it came from (provenance).

In [3]:
class Entity(BaseModel):
    name: str
    type: str = Field(description="e.g. Disease, Treatment, Drug, Test, Symptom, RiskFactor, Organ, Procedure")
    description: str = Field(description="one sentence, grounded in the text")

class Relation(BaseModel):
    source: str
    target: str
    description: str = Field(description="short phrase stating how source relates to target, grounded in the text")

class Extraction(BaseModel):
    entities: list[Entity]
    relations: list[Relation]

EXTRACT_SYS = ("Extract a knowledge graph from a medical patient-guideline passage. List the important entities "
               "(diseases, treatments, drugs, tests, symptoms, risk factors, procedures, organisations, statistics) and "
               "the factual relations between them. Use the most specific canonical name; only state what the text says.")

def extract(i):
    path = f"{CACHE}/extract/{i:04d}.json"
    if os.path.exists(path) and not RUN_LIVE:
        return json.load(open(path))
    ex = chat([{"role": "system", "content": EXTRACT_SYS}, {"role": "user", "content": chunks[i]}],
              "graph index", fmt=Extraction)
    obj = ex.model_dump(); json.dump(obj, open(path, "w")); return obj

os.makedirs(f"{CACHE}/extract", exist_ok=True)
t0 = time.time()
extractions = pmap(extract, range(len(chunks)))
extract_secs = time.time() - t0
TIMING = f"{CACHE}/timings.json"
if extract_secs > 5 or not os.path.exists(TIMING):      # a live extraction ran: remember its wall clock
    json.dump({"extract_secs": extract_secs}, open(TIMING, "w"))
extract_secs = json.load(open(TIMING))["extract_secs"]

norm = lambda s: re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()
G = nx.Graph()
ent_info = {}
for ci, ex in enumerate(extractions):
    G.add_node(("C", ci))
    for e in ex["entities"]:
        k = norm(e["name"])
        if not k:
            continue
        info = ent_info.setdefault(k, {"name": e["name"], "type": e["type"], "descs": [], "chunks": set()})
        info["descs"].append(e["description"]); info["chunks"].add(ci)
        G.add_edge(("E", k), ("C", ci))                      # mention edge (provenance)
    for r in ex["relations"]:
        a, b = norm(r["source"]), norm(r["target"])
        if a and b and a != b:
            for k, nm in ((a, r["source"]), (b, r["target"])):
                ent_info.setdefault(k, {"name": nm, "type": "?", "descs": [], "chunks": set()})["chunks"].add(ci)
                G.add_edge(("E", k), ("C", ci))
            if G.has_edge(("E", a), ("E", b)):
                G[("E", a)][("E", b)]["rels"].append((r["description"], ci))
            else:
                G.add_edge(("E", a), ("E", b), rels=[(r["description"], ci)])

ent_keys = sorted(ent_info)
ent_vecs = cached_npy("entity_vecs.npy", lambda: embed([ent_info[k]["name"] for k in ent_keys], "graph index"))
n_rel = sum(1 for u, v, d in G.edges(data=True) if "rels" in d)
print(f"Extraction: {len(chunks)} calls, {extract_secs:.0f} s wall clock on the live run (16 workers)")
print(f"Graph: {len(ent_keys):,} entities, {n_rel:,} entity-entity relation edges, "
      f"{G.number_of_edges() - n_rel:,} entity-chunk provenance edges")
deg = sorted(((G.degree(("E", k)), ent_info[k]["name"]) for k in ent_keys), reverse=True)[:8]
print("Hub entities:", ", ".join(f"{n} ({d})" for d, n in deg))

Extraction: 519 calls, 181 s wall clock on the live run (16 workers)
Graph: 3,142 entities, 4,820 entity-entity relation edges, 8,073 entity-chunk provenance edges
Hub entities: Chemotherapy (234), Biopsy (188), Radiation Therapy (173), cancer (150), bladder cancer (146), Prostate cancer (142), Cervical Cancer (125), Ovarian cancer (124)


## Two retrievers, one answer prompt

Both systems answer with the same prompt and model; only the context differs. Vector RAG sees the 8 most similar chunks.
Graph RAG links the question's entities into the graph and runs personalised PageRank (damping 0.5, as in HippoRAG),
then passes the 5 best chunks plus up to 25 relation statements among the top-ranked entities.

In [4]:
ANSWER_SYS = ("You answer questions about medical patient guidelines using only the provided context. "
              "Answer directly and completely; do not mention the context.")

def answer(question, context, phase):
    return chat([{"role": "system", "content": ANSWER_SYS},
                 {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}], phase)

class QueryEntities(BaseModel):
    entities: list[str]

def vector_context(qvec, k=8):
    top = np.argsort(-(chunk_vecs @ qvec))[:k]
    return "\n\n".join(f"[chunk {i}] {chunks[i]}" for i in top), [int(i) for i in top]

def graph_context(question, qvec, k_chunks=5, k_rels=25):
    ents = chat([{"role": "system", "content": "List the medical entities (diseases, treatments, tests, symptoms, "
                  "risk factors, ...) mentioned or implied in the question."},
                 {"role": "user", "content": question}], "graph query", fmt=QueryEntities).entities or [question]
    evecs = embed(ents, "graph query")
    pers = defaultdict(float)
    for v in evecs:                                   # link each query entity to its 3 nearest graph entities
        sims = ent_vecs @ v
        for j in np.argsort(-sims)[:3]:
            if sims[j] > 0.5:
                pers[("E", ent_keys[j])] += float(sims[j])
    dense = chunk_vecs @ qvec                          # HippoRAG 2: small weight on dense passage scores
    for j in np.argsort(-dense)[:20]:
        pers[("C", int(j))] += 0.05 * float(dense[j])
    pr = nx.pagerank(G, alpha=0.5, personalization=dict(pers), max_iter=50, tol=1e-6)
    top_c = sorted((n for n in pr if n[0] == "C"), key=lambda n: -pr[n])[:k_chunks]
    top_e = set(sorted((n for n in pr if n[0] == "E"), key=lambda n: -pr[n])[:40])
    rels = []
    for a in top_e:
        for b in G.neighbors(a):
            if b in top_e and b[0] == "E" and a < b:
                for desc, ci in G[a][b]["rels"][:2]:
                    rels.append((pr[a] + pr[b], f"{ent_info[a[1]]['name']} — {desc} — {ent_info[b[1]]['name']}"))
    rel_lines = [r for _, r in sorted(rels, reverse=True)[:k_rels]]
    ctx = ("Relations from the knowledge graph:\n" + "\n".join(f"- {r}" for r in rel_lines) + "\n\n" +
           "\n\n".join(f"[chunk {n[1]}] {chunks[n[1]]}" for n in top_c))
    return ctx, [n[1] for n in top_c]

q_vecs = cached_npy("question_vecs.npy", lambda: embed([q["question"] for q in sample], "queries (shared)"))

def run(i):
    q, path = sample[i], f"{CACHE}/answers/{sample[i]['id']}.json"
    if os.path.exists(path) and not RUN_LIVE:
        return json.load(open(path))
    vctx, vids = vector_context(q_vecs[i])
    gctx, gids = graph_context(q["question"], q_vecs[i])
    rec = {"id": q["id"], "type": q["question_type"],
           "vector": {"answer": answer(q["question"], vctx, "vector query"), "chunks": vids, "ctx_chars": len(vctx)},
           "graph": {"answer": answer(q["question"], gctx, "graph query"), "chunks": gids, "ctx_chars": len(gctx)}}
    json.dump(rec, open(path, "w")); return rec

os.makedirs(f"{CACHE}/answers", exist_ok=True)
t0 = time.time()
runs = pmap(run, range(len(sample)))
print(f"Answered {len(runs)} questions x 2 systems in {time.time() - t0:.0f} s (cached={not RUN_LIVE})")
ex = next(r for r in runs if r["type"] == "Complex Reasoning")
qq = next(q for q in sample if q["id"] == ex["id"])
print(f"\nExample ({ex['type']}): {qq['question']}\n  gold  : {qq['answer']}\n"
      f"  vector: {ex['vector']['answer'][:300]}\n  graph : {ex['graph']['answer'][:300]}")

Answered 180 questions x 2 systems in 0 s (cached=True)

Example (Complex Reasoning): Why is immunophenotyping by flow cytometry essential in the diagnosis of CLL?
  gold  : It identifies CLL cells by their surface proteins (CD20, CD200, LEF1), confirming the diagnosis after initial blood tests.
  vector: Immunophenotyping by flow cytometry is essential in the diagnosis of CLL because it detects specific patterns of proteins, known as biomarkers, on the surface of cells. This method allows for the identification and counting of abnormal white blood cells, specifically monoclonal B lymphocytes, which 
  graph : Immunophenotyping by flow cytometry is essential in the diagnosis of chronic lymphocytic leukemia (CLL) because it detects common patterns of surface proteins on the cells, which helps identify the specific type of cancer. In CLL, the presence of specific proteins such as CD20, CD200, and LEF1 on th


## Scoring with the benchmark's metrics

Re-implemented from the dataset repo's `Evaluation/metrics/` (prompts condensed, logic unchanged):
`answer_correctness` (0.75 × statement F1 + 0.25 × semantic similarity, scaled to [0, 1]), `rouge_score` (ROUGE-L F1) and
`coverage_score` (share of gold facts the answer covers). The judge never sees which system wrote an answer.

In [5]:
class Statements(BaseModel):
    statements: list[str]

class Classified(BaseModel):
    TP: list[str]
    FP: list[str]
    FN: list[str]

class Coverage(BaseModel):
    covered: list[int] = Field(description="1 if the fact is covered by the response, else 0, one per reference fact, in order")

def statements(text):
    return chat([{"role": "user", "content": "Break the text into concise, independent factual statements.\n\n" + text}],
                "evaluation", fmt=Statements).statements

def answer_correctness(question, ans, gold, gold_stmts):
    a_st = statements(ans)
    if not a_st and not gold_stmts:
        f1 = 1.0
    else:
        c = chat([{"role": "user", "content":
                   "Compare answer statements with ground-truth statements. TP: answer statement supported by the "
                   "ground truth. FP: answer statement not supported by (or contradicting) the ground truth. FN: "
                   "ground-truth statement missing from the answer. Return the statements in each list.\n\n"
                   f"Question: {question}\nAnswer statements: {json.dumps(a_st)}\n"
                   f"Ground-truth statements: {json.dumps(gold_stmts)}"}], "evaluation", fmt=Classified)
        tp, fp, fn = len(c.TP), len(c.FP), len(c.FN)
        p, r = tp / (tp + fp + 1e-10), tp / (tp + fn + 1e-10)
        f1 = 2 * p * r / (p + r + 1e-10)
    va, vg = embed([ans, gold], "evaluation")
    return 0.75 * f1 + 0.25 * (float(va @ vg) + 1) / 2

def rouge_l(ans, gold):
    a, g = re.findall(r"\w+", ans.lower()), re.findall(r"\w+", gold.lower())
    if not a or not g:
        return 0.0
    prev = [0] * (len(a) + 1)
    for x in g:                                       # LCS by dynamic programming
        cur = [0]
        for j, y in enumerate(a):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    lcs = prev[-1]
    p, r = lcs / len(a), lcs / len(g)
    return 0 if lcs == 0 else 2 * p * r / (p + r)

def coverage(question, ans, gold_stmts):
    if not gold_stmts:
        return float("nan")
    c = chat([{"role": "user", "content": "For each reference fact, is it covered by the response? Answer 1 or 0 per "
               f"fact, in order.\n\nQuestion: {question}\nResponse: {ans[:3000]}\nReference facts: {json.dumps(gold_stmts)}"}],
             "evaluation", fmt=Coverage).covered
    c = (c + [0] * len(gold_stmts))[:len(gold_stmts)]
    return sum(1 for x in c if x) / len(gold_stmts)

def score(i):
    q, rec = sample[i], runs[i]
    path = f"{CACHE}/scores/{q['id']}.json"
    if os.path.exists(path) and not RUN_LIVE:
        return json.load(open(path))
    gold_stmts = statements(q["answer"])
    out = {"id": q["id"], "type": q["question_type"]}
    for sysname in ("vector", "graph"):
        ans = rec[sysname]["answer"]
        s = {"correctness": answer_correctness(q["question"], ans, q["answer"], gold_stmts)}
        if q["question_type"] in ("Fact Retrieval", "Complex Reasoning"):
            s["rouge_l"] = rouge_l(ans, q["answer"])
        else:
            s["coverage"] = coverage(q["question"], ans, gold_stmts)
        out[sysname] = s
    json.dump(out, open(path, "w")); return out

os.makedirs(f"{CACHE}/scores", exist_ok=True)
t0 = time.time()
scores = pmap(score, range(len(sample)))
print(f"Scored {len(scores)} questions in {time.time() - t0:.0f} s (cached={not RUN_LIVE})")

Scored 180 questions in 0 s (cached=True)


## Results by question type, with 95% bootstrap confidence intervals

The *difference* column is a paired bootstrap over questions (graph − vector); an interval that excludes 0 is a
difference this sample can resolve.

In [6]:
def boot_ci(x, n=5000, seed=7):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    rs = np.random.default_rng(seed)
    m = rs.choice(x, (n, len(x))).mean(1)
    return x.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

fmt = lambda t: f"{100 * t[0]:5.1f} [{100 * t[1]:4.1f}, {100 * t[2]:4.1f}]"
ROWS = {}
print(f"{'type':<22}{'metric':<13}{'vector RAG':>22}{'graph RAG':>22}{'graph - vector':>24}")
for t in TYPES:
    rows = [s for s in scores if s["type"] == t]
    for metric in ("correctness", "rouge_l", "coverage"):
        if metric not in rows[0]["vector"]:
            continue
        v = [r["vector"][metric] for r in rows]; g = [r["graph"][metric] for r in rows]
        d = [b - a for a, b in zip(v, g)]
        ROWS[(t, metric)] = (boot_ci(v), boot_ci(g), boot_ci(d))
        print(f"{t:<22}{metric:<13}{fmt(boot_ci(v)):>22}{fmt(boot_ci(g)):>22}{fmt(boot_ci(d)):>24}")

# Did retrieval reach the gold evidence? Share of gold evidence sentences found (fuzzy, 60% token overlap) in retrieved chunks
def ev_hit(ev, ids):
    toks = set(re.findall(r"\w+", ev.lower()))
    return any(len(toks & set(re.findall(r"\w+", chunks[i].lower()))) >= 0.6 * len(toks) for i in ids)
print("\nGold-evidence recall of the retrieved chunks (vector top-8 vs graph top-5):")
for t in TYPES:
    vr, gr = [], []
    for q, r in zip(sample, runs):
        if q["question_type"] != t or not q.get("evidence"):
            continue
        evs = q["evidence"] if isinstance(q["evidence"], list) else [q["evidence"]]
        vr.append(np.mean([ev_hit(e, r["vector"]["chunks"]) for e in evs]))
        gr.append(np.mean([ev_hit(e, r["graph"]["chunks"]) for e in evs]))
    print(f"  {t:<22} vector {np.mean(vr):5.1%}   graph {np.mean(gr):5.1%}")

type                  metric                   vector RAG             graph RAG          graph - vector
Fact Retrieval        correctness       68.3 [63.4, 73.0]     70.7 [65.5, 75.7]        2.4 [-4.1,  8.7]
Fact Retrieval        rouge_l           29.8 [24.9, 35.5]     35.4 [29.8, 41.8]        5.5 [ 1.8,  9.4]
Complex Reasoning     correctness       72.5 [69.2, 75.9]     73.0 [69.8, 76.2]        0.5 [-3.7,  4.9]
Complex Reasoning     rouge_l           20.0 [17.5, 22.5]     21.0 [18.6, 23.6]        1.0 [-0.8,  2.9]
Contextual Summarize  correctness       71.7 [67.5, 75.6]     73.3 [70.1, 76.4]        1.6 [-3.0,  6.4]
Contextual Summarize  coverage          86.0 [79.5, 91.7]     83.5 [78.3, 88.6]       -2.6 [-7.8,  3.0]
Creative Generation   correctness       68.0 [63.9, 71.9]     66.1 [62.1, 69.9]       -1.9 [-5.5,  2.1]
Creative Generation   coverage          72.2 [67.0, 77.3]     67.6 [62.5, 72.5]       -4.7 [-9.1, -0.1]

Gold-evidence recall of the retrieved chunks (vector top-8 vs g

  Fact Retrieval         vector 99.5%   graph 86.8%


  Complex Reasoning      vector 94.3%   graph 81.6%
  Contextual Summarize   vector 93.5%   graph 80.6%


  Creative Generation    vector 64.0%   graph 49.9%


## The indexing price (Slides 34, 37)

In [7]:
LEDGER_PATH = f"{CACHE}/usage_ledger.json"
if any(u["calls"] for u in USAGE.values()):            # a live run happened in this session: persist it
    old = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {}
    for k, u in USAGE.items():
        o = old.setdefault(k, {"calls": 0, "in": 0, "out": 0})
        for f in u: o[f] += u[f]
    json.dump(old, open(LEDGER_PATH, "w"))
ledger = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else dict(USAGE)
nq = len(sample)
print(f"{'phase':<20}{'calls':>8}{'input tok':>13}{'output tok':>13}")
for k in ("vector index", "graph index", "queries (shared)", "vector query", "graph query", "evaluation"):
    u = ledger.get(k, {"calls": 0, "in": 0, "out": 0})
    print(f"{k:<20}{u['calls']:>8,}{u['in']:>13,}{u['out']:>13,}")
vi, gi = ledger.get("vector index", {}), ledger.get("graph index", {})
vq, gq = ledger.get("vector query", {}), ledger.get("graph query", {})
per_q = lambda u: (u.get("in", 0) + u.get("out", 0)) / nq
print(f"\nIndexing: vector = {vi.get('calls', 0)} embedding calls ({vi.get('in', 0):,} embedding tokens, seconds) | "
      f"graph = {gi.get('calls', 0):,} calls, {gi.get('in', 0):,} input + {gi.get('out', 0):,} output tokens "
      f"(mostly gpt-4o-mini generation, ~{extract_secs:.0f} s with 16 workers on the live run)")
print(f"Per question (answering): vector {per_q(vq):,.0f} tokens | graph {per_q(gq):,.0f} tokens "
      f"({per_q(gq) / max(per_q(vq), 1):.2f}x)")
print(f"Total tokens this experiment (all phases incl. evaluation): "
      f"{sum(u['in'] + u['out'] for u in ledger.values()):,}")

phase                  calls    input tok   output tok
vector index               3      254,540            0
graph index              532      410,997      354,352
queries (shared)           1        4,002            0
vector query             180      722,351       31,662
graph query              540      563,857       33,917
evaluation             1,420      401,986      139,119

Indexing: vector = 3 embedding calls (254,540 embedding tokens, seconds) | graph = 532 calls, 410,997 input + 354,352 output tokens (mostly gpt-4o-mini generation, ~181 s with 16 workers on the live run)
Per question (answering): vector 4,189 tokens | graph 3,321 tokens (0.79x)
Total tokens this experiment (all phases incl. evaluation): 2,916,783


In [8]:
PAPER = {"Fact Retrieval": (60.92, 49.29), "Complex Reasoning": (42.93, 53.38), "Contextual Summarize": (51.30, 64.40)}
print("=" * 86)
print(f"RESULTS SUMMARY — Notebook 9: GraphRAG-Bench Medical, {nq} questions, gpt-4o-mini, answer correctness (x100)")
print("=" * 86)
print(f"{'question type':<24}{'n':>4}{'vector RAG':>13}{'graph RAG':>12}{'diff [95% CI]':>22}{'paper: RAG / best graph':>26}")
for t, n in TYPES.items():
    v, g, d = ROWS[(t, "correctness")]
    paper = PAPER.get(t)
    ptxt = f"{paper[0]:.2f} / {paper[1]:.2f}" if paper else "-"
    print(f"{t:<24}{n:>4}{100 * v[0]:>13.1f}{100 * g[0]:>12.1f}{f'{100 * d[0]:+.1f} [{100 * d[1]:+.1f}, {100 * d[2]:+.1f}]':>22}{ptxt:>26}")
print(f"\nGraph: {len(ent_keys):,} entities, {n_rel:,} relation edges from {len(chunks)} chunks; indexing "
      f"{gi.get('calls', 0):,} LLM/embedding calls ({(gi.get('in', 0) + gi.get('out', 0)) / 1e6:.2f} M tokens, "
      f"{gi.get('out', 0):,} of them generated) vs {vi.get('calls', 0)} embedding calls; answering "
      f"{per_q(gq) / max(per_q(vq), 1):.2f}x the tokens per question")
print("\nLecture message (Slides 36-38): on this corpus the graph index bought no resolvable accuracy — every"
      "\ncorrectness difference has a 95% CI spanning 0, and vector top-8 already retrieved 94-99% of the gold"
      "\nevidence for fact, reasoning and summary questions. The paper's 11-13 point gaps did not reproduce at this"
      "\nscale with this graph method. The graph is worth its indexing price only where the baseline actually"
      "\nfails — measure the baseline by question type first (slide 68).")

RESULTS SUMMARY — Notebook 9: GraphRAG-Bench Medical, 180 questions, gpt-4o-mini, answer correctness (x100)
question type              n   vector RAG   graph RAG         diff [95% CI]   paper: RAG / best graph
Fact Retrieval            50         68.3        70.7     +2.4 [-4.1, +8.7]             60.92 / 49.29
Complex Reasoning         50         72.5        73.0     +0.5 [-3.7, +4.9]             42.93 / 53.38
Contextual Summarize      50         71.7        73.3     +1.6 [-3.0, +6.4]             51.30 / 64.40
Creative Generation       30         68.0        66.1     -1.9 [-5.5, +2.1]                         -

Graph: 3,142 entities, 4,820 relation edges from 519 chunks; indexing 532 LLM/embedding calls (0.77 M tokens, 354,352 of them generated) vs 3 embedding calls; answering 0.79x the tokens per question

Lecture message (Slides 36-38): on this corpus the graph index bought no resolvable accuracy — every
correctness difference has a 95% CI spanning 0, and vector top-8 already retriev